# 06 · Temporal aggregation, trend quality, escalation rule & latency

Implements paper Sec. 5.6–5.7 and 6.4–6.5.

1. Load the **final model** (`models/final_model.json`; falls back to the classical Logistic Regression if no transformer is trained yet).
2. Build **simulated user timelines** from scripted emotion mixtures (no public dataset has per-user timelines — state this in the report).
3. Store timestamped predictions in **SQLite** (derived scores only, **no raw text** — privacy by design).
4. Aggregate to daily / weekly / monthly mood summaries: dominant emotion `dom(W)`, negative-affect score `S(W)`, EMA-smoothed `S̃_t`.
5. **Trend quality**: compare aggregated trends with the scripted ground truth, and aggregated vs. single-message signals (RQ4).
6. **Escalation rule** `S̃_t > τ for ≥ k consecutive days`: calibrate τ, k on *validation* timelines, evaluate on *test* timelines (sustained vs. isolated negativity).
7. **Latency** of the end-to-end pipeline (preprocessing → inference → aggregation).

**Outputs:** `results/trend_quality.csv`, `results/escalation_eval.csv`, `results/trend_params.json`, `results/latency_results.csv`, `data/processed/mood_demo.db`, figures.

In [ ]:
!pip -q install torch transformers scikit-learn seaborn emoji nltk joblib

In [ ]:
import os, sys, json, random, time, warnings, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# ---- Project root: Google Drive folder on Colab, repo root locally ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/mental-health-nlp")   # <- change if you keep the repo elsewhere
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for d in ["data/raw", "data/interim", "data/processed", "data/splits",
          "models/classical", "models/bilstm", "models/distilbert", "models/roberta",
          "results/figures", "results/reports"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SEED = 42
EMOTIONS = ["Joy", "Sadness", "Anger", "Fear", "Anxiety", "Stress", "Surprise", "Neutral"]
LABEL2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2LABEL = {i: e for e, i in LABEL2ID.items()}
NEGATIVE = ["Sadness", "Anger", "Fear", "Anxiety", "Stress"]      # N in the paper (Sec. 5.6)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()
print("ROOT =", ROOT)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             confusion_matrix, classification_report)

METRICS_CSV = ROOT / "results/model_metrics.csv"
LATENCY_CSV = ROOT / "results/latency_results.csv"
LABEL_IDS = list(range(len(EMOTIONS)))

def slug(condition):
    """File-system friendly condition name ('augment+class_weight' -> 'augment-class_weight')."""
    return condition.replace("+", "-")

def _upsert(path, row, keys):
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(df):
        mask = np.ones(len(df), dtype=bool)
        for k in keys:
            mask &= (df[k].astype(str) == str(row[k])).values
        df = df[~mask]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)

def evaluate(y_true, y_pred, model, condition="baseline", seed=SEED, extra=None, log=True):
    """Compute all metrics, optionally upsert into results/model_metrics.csv."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABEL_IDS,
                                                 average="macro", zero_division=0)
    per = f1_score(y_true, y_pred, labels=LABEL_IDS, average=None, zero_division=0)
    row = dict(model=model, condition=condition, seed=seed,
               accuracy=accuracy_score(y_true, y_pred),
               precision_macro=p, recall_macro=r, macro_f1=f, min_class_f1=per.min())
    row.update({f"f1_{e}": v for e, v in zip(EMOTIONS, per)})
    if extra:
        row.update(extra)
    if log:
        _upsert(METRICS_CSV, row, ["model", "condition", "seed"])
    return row

def log_latency(model, device, ms_per_text, note=""):
    _upsert(LATENCY_CSV, dict(model=model, device=device, ms_per_text=round(float(ms_per_text), 2), note=note),
            ["model", "device", "note"])

def plot_cm(y_true, y_pred, title, path=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)
    if normalize:
        cm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax, cbar=False)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()

In [ ]:
SPLIT_DIR = ROOT / "data/splits"
train = pd.read_csv(SPLIT_DIR / "train.csv")
val   = pd.read_csv(SPLIT_DIR / "val.csv")
test  = pd.read_csv(SPLIT_DIR / "test.csv")
for d in (train, val, test):
    d["text_tf"] = d["text_tf"].fillna("").astype(str)
    d["text_clean"] = d["text_clean"].fillna("").astype(str)
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})
train["label"].value_counts().reindex(EMOTIONS)

In [ ]:
import re, emoji, nltk
from functools import lru_cache
for _pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(_pkg, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

URL_RE    = re.compile(r"https?://\S+|www\.\S+")
HTML_RE   = re.compile(r"<[^>]+>|&\w+;")
HANDLE_RE = re.compile(r"(?<!\w)@\w+|(?<!\w)/?u/\w+")
REPEAT_RE = re.compile(r"([!?.])\1{2,}")
WS_RE     = re.compile(r"\s+")
TOKEN_RE  = re.compile(r"[a-z]+(?:'[a-z]+)?")

# Negations are KEPT even in the classical pipeline ("not happy" must not become "happy").
_sw = set(stopwords.words("english"))
_NEG_BASE = {"no", "nor", "not", "never", "none", "nothing", "neither", "nobody", "cannot",
             "don", "didn", "doesn", "isn", "wasn", "aren", "couldn", "shouldn", "wouldn",
             "hasn", "haven", "hadn", "won", "weren", "needn", "mustn", "shan", "mightn", "ain"}
NEGATIONS  = {w for w in _sw if w in _NEG_BASE or "n't" in w} | _NEG_BASE
STOP_WORDS = _sw - NEGATIONS
_lemm = WordNetLemmatizer()

def preprocess_tf(text):
    """Light cleaning for transformers / Bi-LSTM: no lowercasing, no stop-word removal, no lemmatisation."""
    t = str(text)
    t = URL_RE.sub(" ", t); t = HTML_RE.sub(" ", t); t = HANDLE_RE.sub(" ", t)
    t = emoji.demojize(t, delimiters=(" ", " ")).replace("_", " ")      # emoji -> emotion-bearing tokens
    t = REPEAT_RE.sub(r"\1\1", t)                                       # punctuation normalisation
    return WS_RE.sub(" ", t).strip()

@lru_cache(maxsize=200_000)
def _lemma(w):
    return _lemm.lemmatize(w)

def preprocess_classical(text):
    """Full pipeline for TF-IDF models: lowercase, strip punctuation, drop stop-words (not negations), lemmatise."""
    t = preprocess_tf(text).lower().replace("\u2019", "'")
    return " ".join(_lemma(w) for w in TOKEN_RE.findall(t) if w not in STOP_WORDS)

## 1 · Load the predictor (message text → probability distribution over the 8 emotions)

In [ ]:
import joblib, torch

class Predictor:
    """f: X -> Δ^|E|.  Uses the promoted transformer if available, else the classical LR baseline."""
    def __init__(self):
        self.model = None
        meta = ROOT / "models/final_model.json"
        keys = ([json.loads(meta.read_text())["model"]] if meta.exists() else []) + ["distilbert", "roberta"]
        for key in keys:
            p = ROOT / "models" / key / "final"
            if (p / "config.json").exists():
                from transformers import AutoTokenizer, AutoModelForSequenceClassification
                self.tok = AutoTokenizer.from_pretrained(str(p))
                self.model = AutoModelForSequenceClassification.from_pretrained(str(p)).eval()
                self.kind, self.name = "transformer", key
                self.to("cuda" if torch.cuda.is_available() else "cpu")
                break
        if self.model is None:
            self.pipe = joblib.load(ROOT / "models/classical/logreg_baseline.joblib")
            self.kind, self.name, self.device = "classical", "logreg", "cpu"
        print(f"Predictor: {self.name} ({self.kind}) on {self.device}")

    def to(self, dev):
        self.device = dev; self.model.to(dev); return self

    def _proba(self, texts):
        if self.kind == "classical":
            return self.pipe.predict_proba(texts)
        out = []
        for i in range(0, len(texts), 128):
            enc = self.tok(texts[i:i + 128], truncation=True, max_length=64, padding=True, return_tensors="pt").to(self.device)
            with torch.no_grad():
                out.append(torch.softmax(self.model(**enc).logits, -1).cpu().numpy())
        return np.vstack(out)

    def predict_df(self, df):            # uses pre-computed text views
        return self._proba(df["text_clean" if self.kind == "classical" else "text_tf"].tolist())

    def predict_raw(self, texts):        # full pipeline incl. preprocessing (used for latency)
        fn = preprocess_classical if self.kind == "classical" else preprocess_tf
        return self._proba([fn(t) for t in texts])

predictor = Predictor()

## 2 · Scripted timelines
Each scenario defines, for every day, a **mixture over the 8 emotions**. A message is generated by sampling an emotion from that day's mixture and drawing a real text of that emotion from the chosen pool (validation pool for calibration, test pool for evaluation — so the model never sees them in training).
The ground-truth negative-affect share for the day is the mixture's mass on {Sadness, Anger, Fear, Anxiety, Stress}.

In [ ]:
NEG_IDX = [LABEL2ID[e] for e in NEGATIVE]
PCOLS = [f"p_{e}" for e in EMOTIONS]
NEG_PCOLS = [f"p_{e}" for e in NEGATIVE]

def mix(**w):
    v = np.zeros(len(EMOTIONS))
    for k, x in w.items(): v[LABEL2ID[k]] = x
    return v / v.sum()

POS  = mix(Joy=.50, Neutral=.35, Surprise=.15)
NEGM = mix(Sadness=.30, Stress=.25, Anxiety=.20, Anger=.10, Fear=.05, Neutral=.10)
lerp = lambda a, b, w: (1 - w) * a + w * b

SCENARIOS = {                                   # day d of n  ->  mixture over emotions
    "stable_positive":    lambda d, n: POS,
    "drift_to_negative":  lambda d, n: lerp(POS, NEGM, d / (n - 1)),     # e.g. Joy drifting to Sadness/Stress over weeks
    "sustained_negative": lambda d, n: NEGM,
    "recovery":           lambda d, n: lerp(NEGM, POS, d / (n - 1)),
    "isolated_spike":     lambda d, n: NEGM if d in (n // 2, n // 2 + 1) else POS,   # 2 bad days only -> must NOT escalate
}

def make_timeline(name, pool, n_days=42, seed=0, start="2026-01-01"):
    rng = np.random.default_rng(seed)
    by_c = {c: pool[pool.label_id == c].reset_index(drop=True) for c in range(len(EMOTIONS))}
    rows = []
    for d in range(n_days):
        w = SCENARIOS[name](d, n_days)
        for _ in range(int(rng.integers(1, 5))):                       # 1-4 messages per day
            c = int(rng.choice(len(EMOTIONS), p=w))
            r = by_c[c].iloc[int(rng.integers(len(by_c[c])))]
            ts = pd.Timestamp(start) + pd.Timedelta(days=d, hours=int(rng.integers(8, 23)), minutes=int(rng.integers(0, 60)))
            rows.append(dict(user_id=f"{name}_{seed}", scenario=name, timestamp=ts, day=d, true_label=c,
                             script_neg=float(w[NEG_IDX].sum()), script_dom=int(w.argmax()),
                             text_tf=r["text_tf"], text_clean=r["text_clean"]))
    return pd.DataFrame(rows)

def build_and_predict(pool, n_per=10, n_days=42, seed0=0):
    tl = pd.concat([make_timeline(sc, pool, n_days, seed0 + i) for sc in SCENARIOS for i in range(n_per)], ignore_index=True)
    P = predictor.predict_df(tl)
    tl[PCOLS] = P
    tl["pred"] = P.argmax(1); tl["conf"] = P.max(1)
    tl["s"] = tl[NEG_PCOLS].sum(axis=1)                    # per-message negative-affect score s_i
    return tl

t0 = time.time()
TL_VAL  = build_and_predict(val,  n_per=10, seed0=1000)    # for calibrating tau, k
TL_TEST = build_and_predict(test, n_per=10, seed0=2000)    # for final reporting
print(f"{len(TL_VAL)} val msgs, {len(TL_TEST)} test msgs  ({time.time()-t0:.0f}s)")
print("message-level accuracy vs. sampled label (val / test):",
      round((TL_VAL.pred == TL_VAL.true_label).mean(), 3), "/", round((TL_TEST.pred == TL_TEST.true_label).mean(), 3))

## 3 · Store timestamped predictions in SQLite (derived scores only)
Schema: `(user_id, timestamp, emotion, confidence, sentiment_score, probs)` — raw text is **not** stored (data minimisation, paper Sec. 10.2).
`src/database.py` can wrap exactly this.

In [ ]:
import sqlite3
DB = ROOT / "data/processed/mood_demo.db"
if DB.exists(): DB.unlink()
con = sqlite3.connect(DB)
con.execute("""CREATE TABLE predictions (user_id TEXT, timestamp TEXT, emotion TEXT, confidence REAL,
                                         sentiment_score REAL, probs TEXT)""")
demo = TL_TEST[TL_TEST.user_id == "drift_to_negative_2000"]
con.executemany("INSERT INTO predictions VALUES (?,?,?,?,?,?)",
                [(r.user_id, r.timestamp.isoformat(), ID2LABEL[r.pred], float(r.conf), float(r.s),
                  json.dumps([round(float(x), 4) for x in r[PCOLS].values])) for _, r in demo.iterrows()])
con.commit()
back = pd.read_sql("SELECT * FROM predictions", con, parse_dates=["timestamp"])
print(len(back), "rows stored for demo user"); display(back.head(3)); con.close()

## 4 · Aggregation: daily, weekly, monthly
* `dom(W) = argmax_e mean_{i∈W} p_i(e)`
* `s_i = Σ_{e∈N} p_i(e)`, `S(W) = mean_{i∈W} s_i`
* `S̃_t = α·S_t + (1−α)·S̃_{t−1}` (EMA, α=0.3 initial setting)

In [ ]:
ALPHA = 0.3

def daily_table(g, alpha=ALPHA):
    d = g.groupby("day").agg(S=("s", "mean"), s_max=("s", "max"), n_msgs=("s", "size"),
                             script_neg=("script_neg", "first"), script_dom=("script_dom", "first"),
                             **{c: (c, "mean") for c in PCOLS})
    d["S_ema"] = d["S"].ewm(alpha=alpha, adjust=False).mean()
    d["dominant"] = d[PCOLS].idxmax(axis=1).str[2:]
    d["script_dom_name"] = d["script_dom"].map(ID2LABEL)
    return d

def window_table(g, rule):
    """Weekly ('7D') or monthly ('MS') summaries computed from message-level probabilities."""
    w = g.set_index("timestamp")[PCOLS + ["s"]].resample(rule).mean()
    w["dominant"] = w[PCOLS].idxmax(axis=1).str[2:]
    return w[["dominant", "s"]].rename(columns={"s": "S"})

def all_daily(tl):
    return {u: daily_table(g) for u, g in tl.groupby("user_id")}

DAILY_VAL, DAILY_TEST = all_daily(TL_VAL), all_daily(TL_TEST)

g = TL_TEST[TL_TEST.user_id == "drift_to_negative_2000"]
print("Weekly:");  display(window_table(g, "7D").round(3))
print("Monthly:"); display(window_table(g, "MS").round(3))

## 5 · Trend quality vs. scripted ground truth (and aggregated vs. single-message — RQ4)

In [ ]:
def safe_corr(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.nan if a.std() < 1e-9 or b.std() < 1e-9 else float(np.corrcoef(a, b)[0, 1])

def trend_report(tl, daily):
    rows = []
    for sc in SCENARIOS:
        msg = tl[tl.scenario == sc]
        D = pd.concat([daily[u] for u in msg.user_id.unique()])
        rows.append(dict(
            scenario=sc,
            dominant_exact=(D.dominant == D.script_dom_name).mean(),                       # e.g. Stress vs Sadness counts as a miss
            valence_agree=(D.dominant.isin(NEGATIVE) == D.script_dom_name.isin(NEGATIVE)).mean(),   # negative vs non-negative
            mae_message=(msg.s - msg.script_neg).abs().mean(),
            mae_daily=(D.S - D.script_neg).abs().mean(),
            mae_ema=(D.S_ema - D.script_neg).abs().mean(),
            r_message=safe_corr(msg.s, msg.script_neg),
            r_daily=safe_corr(D.S, D.script_neg),
            r_ema=safe_corr(D.S_ema, D.script_neg)))
    return pd.DataFrame(rows).set_index("scenario")

TQ = trend_report(TL_TEST, DAILY_TEST)
display(TQ.round(3))
print("\nRQ4 check — mean absolute error w.r.t. scripted negative share (lower is better):")
print(TQ[["mae_message", "mae_daily", "mae_ema"]].mean().round(3).to_dict())
print("Correlation on gradual scenarios (drift/recovery):")
display(TQ.loc[["drift_to_negative", "recovery"], ["r_message", "r_daily", "r_ema"]].round(3))
TQ.round(4).to_csv(ROOT / "results/trend_quality.csv")

## 6 · Escalation rule — calibrate on validation timelines, test on held-out timelines
**Rule R:** escalate when `S̃_t > τ` for at least `k` consecutive days.

**Ground truth (from the script, not from the classifier):** a day is a "should-escalate" day if the scripted negative share has been ≥ 0.6 for the last 5 days. This makes *sustained* negativity positive and an *isolated 2-day spike* negative.

In [ ]:
def escalate(series, tau, k):
    run, flags = 0, []
    for v in np.asarray(series, float):
        run = run + 1 if v > tau else 0
        flags.append(run >= k)
    return np.array(flags)

TRUTH_TAU, TRUTH_K = 0.6, 5

def eval_flags(daily, flag_fn):
    tp = fp = fn = tn = 0; delays = []; false_alarm_tl = missed_tl = 0
    for u, D in daily.items():
        pred = flag_fn(D); truth = escalate(D.script_neg.values, TRUTH_TAU - 1e-9, TRUTH_K)
        tp += int((pred & truth).sum()); fp += int((pred & ~truth).sum())
        fn += int((~pred & truth).sum()); tn += int((~pred & ~truth).sum())
        if pred.any() and not truth.any(): false_alarm_tl += 1
        if truth.any() and not pred.any(): missed_tl += 1
        if pred.any() and truth.any(): delays.append(int(np.argmax(pred) - np.argmax(truth)))
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(precision=prec, recall=rec, f1=2 * prec * rec / max(prec + rec, 1e-9), day_fpr=fp / max(fp + tn, 1),
                false_alarm_timelines=false_alarm_tl, missed_timelines=missed_tl, n_timelines=len(daily),
                median_delay_days=float(np.median(delays)) if delays else np.nan)

grid = []
for tau in np.round(np.arange(0.30, 0.76, 0.05), 2):
    for k in [2, 3, 5, 7]:
        r = eval_flags(DAILY_VAL, lambda D, tau=tau, k=k: escalate(D.S_ema.values, tau, k))
        grid.append(dict(tau=tau, k=k, **r))
grid = pd.DataFrame(grid).sort_values(["f1", "false_alarm_timelines"], ascending=[False, True])
display(grid.head(8).round(3))
TAU, K = float(grid.iloc[0].tau), int(grid.iloc[0].k)
print(f"\nCalibrated on validation timelines: tau={TAU}, k={K}, alpha={ALPHA}")

In [ ]:
# Final evaluation on held-out TEST timelines + ablation against a single-message alarm (no persistence requirement)
res = {
    f"EMA rule (tau={TAU}, k={K})":  eval_flags(DAILY_TEST, lambda D: escalate(D.S_ema.values, TAU, K)),
    "daily-mean rule (no smoothing)": eval_flags(DAILY_TEST, lambda D: escalate(D.S.values, TAU, K)),
    "single-message alarm (s_i > 0.8)": eval_flags(DAILY_TEST, lambda D: D.s_max.values > 0.8),
}
ESC = pd.DataFrame(res).T
display(ESC.round(3)); ESC.round(4).to_csv(ROOT / "results/escalation_eval.csv")

# per-scenario: fraction of timelines that ever escalate (should be high for sustained/drift, ~0 for spike/stable)
rows = []
for sc in SCENARIOS:
    us = TL_TEST[TL_TEST.scenario == sc].user_id.unique()
    ever = [escalate(DAILY_TEST[u].S_ema.values, TAU, K).any() for u in us]
    truth = [escalate(DAILY_TEST[u].script_neg.values, TRUTH_TAU - 1e-9, TRUTH_K).any() for u in us]
    rows.append(dict(scenario=sc, timelines=len(us), escalated=int(sum(ever)), should_escalate=int(sum(truth))))
display(pd.DataFrame(rows))

json.dump({"alpha": ALPHA, "tau": TAU, "k": K, "truth_tau": TRUTH_TAU, "truth_k": TRUTH_K, "negative_emotions": NEGATIVE,
           "predictor": predictor.name, "calibrated_on": "simulated validation timelines"},
          open(ROOT / "results/trend_params.json", "w"), indent=2)

## 7 · Visualising the trend
Dots = per-message `s_i`, solid line = EMA `S̃_t`, dashed grey = scripted negative share, red dashed = threshold τ, red shading = days on which the rule escalates.

In [ ]:
fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(11, 3.0 * len(SCENARIOS)), sharex=True)
for ax, sc in zip(axes, SCENARIOS):
    u = TL_TEST[TL_TEST.scenario == sc].user_id.iloc[0]
    g, D = TL_TEST[TL_TEST.user_id == u], DAILY_TEST[u]
    flags = escalate(D.S_ema.values, TAU, K)
    ax.scatter(g.day, g.s, s=12, alpha=.4, label="per-message s_i")
    ax.plot(D.index, D.S_ema, lw=2.2, label="EMA  S̃_t")
    ax.plot(D.index, D.script_neg, "--", color="gray", label="scripted negative share")
    ax.axhline(TAU, color="crimson", ls=":", label=f"τ = {TAU}")
    for d in D.index[flags]: ax.axvspan(d - .5, d + .5, color="crimson", alpha=.12, lw=0)
    ax.set_ylim(-0.02, 1.02); ax.set_title(sc, loc="left", fontsize=10); ax.set_ylabel("negative affect")
axes[0].legend(ncol=4, fontsize=8, loc="upper right"); axes[-1].set_xlabel("day")
plt.tight_layout(); plt.savefig(ROOT / "results/figures/06_trends_by_scenario.png", dpi=150); plt.show()

## 8 · Rule-based recommendations (prototype → port to `src/recommend.py`)
Transparent, auditable, **non-diagnostic** wording. Verify helpline numbers for the user's region before shipping (e.g. India: Tele-MANAS **14416** — confirm it is still current).

In [ ]:
SUGGESTIONS = {
    "Stress":  "Stress seems to be showing up in your recent entries. A slow breathing exercise (inhale 4s, hold 4s, exhale 6s, a few rounds) can help.",
    "Anxiety": "Things may feel on edge lately. A short grounding exercise - name 5 things you see, 4 you feel, 3 you hear - can help.",
    "Sadness": "Your recent entries sound heavy. Talking to someone you trust, or browsing some curated self-care resources, might help.",
    "Anger":   "There seems to be some frustration lately. A short walk or writing down what is bothering you can create a bit of space.",
    "Fear":    "Worry seems present in your entries. Naming the specific worry and one small next step can make it feel more manageable.",
}
ESCALATION_TEXT = ("Over the past days your entries have shown a persistent low mood. This tool cannot diagnose anything, "
                   "but talking with a qualified mental-health professional could really help. "
                   "Helpline: <insert verified local number>. If you are in immediate danger, contact local emergency services.")

def recommend(day_row, escalated):
    if escalated: return ESCALATION_TEXT
    if day_row["dominant"] in SUGGESTIONS and day_row["S_ema"] > 0.4: return SUGGESTIONS[day_row["dominant"]]
    return "No action needed."

D = DAILY_TEST["drift_to_negative_2000"]; fl = escalate(D.S_ema.values, TAU, K)
for d in [5, 20, 30, 41]:
    print(f"day {d:2d} | dominant={D.iloc[d].dominant:8s} S_ema={D.iloc[d].S_ema:.2f} escalated={bool(fl[d])}\n   -> {recommend(D.iloc[d], fl[d])}\n")

## 9 · End-to-end latency (preprocessing → inference → aggregation)

In [ ]:
sample_texts = test["text"].sample(50, random_state=SEED).tolist()
dummy = TL_TEST[TL_TEST.user_id == "drift_to_negative_2000"]

def end_to_end(text):
    p = predictor.predict_raw([text])                                   # preprocess + inference
    row = pd.DataFrame(p, columns=PCOLS); row["s"] = row[NEG_PCOLS].sum(axis=1)
    _ = daily_table(pd.concat([dummy.tail(30)[["day", "s", "script_neg", "script_dom"] + PCOLS], row.assign(day=99, script_neg=0, script_dom=0)]))
    return p

def bench(fn, n=len(sample_texts)):
    fn(sample_texts[0])                                                 # warm-up
    t0 = time.perf_counter()
    for t in sample_texts[:n]: fn(t)
    return 1000 * (time.perf_counter() - t0) / n

devices = [predictor.device] if predictor.kind == "classical" else ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
for dev in devices:
    if predictor.kind == "transformer": predictor.to(dev)
    e2e, inf = bench(end_to_end), bench(lambda t: predictor.predict_raw([t]))
    tag = "gpu" if dev == "cuda" else "cpu"
    log_latency(predictor.name, tag, e2e, note="end-to-end: preprocess + inference + daily aggregation, batch=1")
    print(f"{predictor.name} on {dev}: end-to-end {e2e:.1f} ms | preprocess+inference {inf:.1f} ms")
print("\n(Dashboard rendering latency is measured separately in the app, e.g. with browser dev-tools.)")
display(pd.read_csv(LATENCY_CSV))

### ✅ Done
Artifacts: `results/trend_quality.csv`, `results/escalation_eval.csv`, `results/trend_params.json` (τ, k, α for the app), `data/processed/mood_demo.db`.

**State clearly in the report:** timelines are *simulated*; trend quality and escalation tests verify that the module behaves *as specified* on constructed sequences — they do **not** establish clinical validity.